# Phase 3 — Sentiment vs. Price Movement Validation

Qualitative check: overlay daily sentiment scores against price returns for a few stocks.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

from data.storage.db_client import load_ohlcv, load_daily_sentiment

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
TICKERS = ['AAPL', 'TSLA', 'JPM', 'XOM', 'NVDA']
START = '2023-01-01'

In [ ]:
def plot_sentiment_vs_price(ticker, start=START):
    price_df = load_ohlcv(ticker, start=start)
    sent_df = load_daily_sentiment(ticker, start=start)

    if price_df.empty:
        print(f'No price data for {ticker}')
        return
    if sent_df.empty:
        print(f'No sentiment data for {ticker}')
        return

    price_df['daily_return'] = price_df['adj_close'].pct_change()
    # Rolling 5-day smoothed sentiment
    sent_df['avg_score_smooth'] = sent_df['avg_score'].rolling(5, min_periods=1).mean()

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                                         gridspec_kw={'height_ratios': [3, 2, 1]})
    fig.suptitle(f'{ticker} — Sentiment vs. Price', fontsize=14, fontweight='bold')

    # Price
    ax1.plot(price_df.index, price_df['adj_close'], color='#2196F3', linewidth=1.2)
    ax1.set_ylabel('Adj Close ($)')
    ax1.grid(True, alpha=0.3)

    # Sentiment
    colors = sent_df['avg_score'].apply(
        lambda x: '#4CAF50' if x > 0.15 else ('#F44336' if x < -0.15 else '#9E9E9E')
    )
    ax2.bar(sent_df.index, sent_df['avg_score'], color=colors, alpha=0.6, width=1)
    ax2.plot(sent_df.index, sent_df['avg_score_smooth'], color='#FF9800',
             linewidth=1.5, label='5-day MA')
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Sentiment Score')
    ax2.set_ylim(-1, 1)
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    # Article count
    ax3.bar(sent_df.index, sent_df['article_count'], color='#9C27B0', alpha=0.5, width=1)
    ax3.set_ylabel('# Articles')
    ax3.set_xlabel('Date')
    ax3.grid(True, alpha=0.3)

    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [ ]:
for ticker in TICKERS:
    plot_sentiment_vs_price(ticker)

## Correlation Analysis

Check lead/lag correlations between sentiment and next-day returns.

In [ ]:
def sentiment_return_correlation(ticker, start=START):
    price_df = load_ohlcv(ticker, start=start)
    sent_df = load_daily_sentiment(ticker, start=start)

    if price_df.empty or sent_df.empty:
        return None

    price_df['daily_return'] = price_df['adj_close'].pct_change()
    price_df['next_day_return'] = price_df['daily_return'].shift(-1)

    merged = price_df[['daily_return', 'next_day_return']].join(sent_df[['avg_score']], how='inner')
    merged = merged.dropna()

    if len(merged) < 20:
        return None

    return {
        'ticker': ticker,
        'n_days': len(merged),
        'corr_same_day': merged['avg_score'].corr(merged['daily_return']),
        'corr_next_day': merged['avg_score'].corr(merged['next_day_return']),
    }

results = []
for ticker in TICKERS:
    r = sentiment_return_correlation(ticker)
    if r:
        results.append(r)

if results:
    corr_df = pd.DataFrame(results).set_index('ticker')
    print(corr_df.to_string())
else:
    print('No data available — run the sentiment pipeline first.')

## Filing Sentiment — Year-over-Year Shifts

In [ ]:
from sqlalchemy import text as sql_text
from data.storage.db_client import get_conn

sql = sql_text("""
    SELECT ticker, period, avg_score, prev_period_score, score_delta, label
    FROM filing_sentiment
    ORDER BY ticker, period
""")

with get_conn() as conn:
    filing_sent_df = pd.read_sql(sql, conn, parse_dates=['period'])

if not filing_sent_df.empty:
    for ticker in filing_sent_df['ticker'].unique()[:5]:
        sub = filing_sent_df[filing_sent_df['ticker'] == ticker]
        print(f'\n{ticker}:')
        print(sub[['period', 'avg_score', 'score_delta', 'label']].to_string(index=False))
else:
    print('No filing sentiment data — run the pipeline first.')